# Week 28, Day 1: Scales of the brain: from molecules to behaviour

Companion notebook for the Engram lesson at https://engram-neuroai.vercel.app/learn/w028/d1.
Run the cells in order. Exercises have a checks cell after them: run it to test your answer.

### Exercise: How many synapses?

Draw 100,000 samples of the number of neurons (log-uniform between 80 and 90 billion) and of synapses per neuron (log-uniform between 1,000 and 10,000), using a local rng with seed 28. Multiply them into `synapses` and report the median and 90% interval.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
r = np.random.default_rng(28)

def log_uniform(lo, hi, n):
    return 10 ** r.uniform(np.log10(lo), np.log10(hi), n)

neurons = log_uniform(80e9, 90e9, 100000)
per_neuron = ...
synapses = ...
lo, med, hi = np.percentile(synapses, [5, 50, 95])
print(f"synapses: median {med:.1e}, 90% interval {lo:.1e} to {hi:.1e}")
plt.hist(np.log10(synapses), 60); plt.xlabel("log10 synapses"); plt.show()

In [ ]:
import numpy as np
def test_samples():
    "synapses holds 100,000 samples of neurons times synapses per neuron"
    assert synapses.shape == (100000,) and np.all(synapses > 0)

def test_range():
    "the median is between 1e14 and 1e15, and the 90% interval spans less than a factor of 20"
    lo, med, hi = np.percentile(synapses, [5, 50, 95])
    assert 1e14 < med < 1e15 and hi / lo < 20

test_samples()
test_range()
print("All checks passed")

### Exercise: How fast can neurons fire on average?

Suppose 20 W of power, half of it (`frac_signal` = 0.5) available for spikes and synaptic transmission, 8e-20 J per ATP and 1e9 ATP per spike. Compute the ATP molecules per second available for signalling, `atp_per_s`, and the largest average firing rate across 86 billion neurons, `max_rate`. Then compute it for spike costs of 3e8 and 3e9 ATP.

In [ ]:
power, frac_signal, e_atp = 20.0, 0.5, 8e-20
n_neurons, atp_per_spike = 86e9, 1e9
atp_per_s = ...
max_rate = ...
for cost in (3e8, 3e9):
    print(f"cost {cost:.0e} ATP per spike: average rate at most {atp_per_s / (n_neurons * cost):.2f} spikes/s")
print(f"{atp_per_s:.2e} ATP per second for signalling;  average rate at most {max_rate:.2f} spikes/s")

In [ ]:
import numpy as np
def test_atp():
    "atp_per_s is the share of the power for signalling divided by the energy per ATP"
    assert abs(atp_per_s - frac_signal * power / e_atp) / atp_per_s < 1e-12

def test_rate():
    "max_rate is the ATP rate divided by neurons times ATP per spike, and is a few spikes per second or less"
    assert abs(max_rate - atp_per_s / (n_neurons * atp_per_spike)) < 1e-12 and 0.1 < max_rate < 5

test_atp()
test_rate()
print("All checks passed")

### Exercise: Two scaling laws

Fit a power law, log neurons = a + b log mass, separately to the three primates and the two rodents below, and store the exponents b in `slope['primate']` and `slope['rodent']`. Then predict the neurons in a rodent brain as heavy as a human's by extrapolating the rodent law.

In [ ]:
mass = np.array([7.8, 87, 1508, 0.4, 1.8])          # marmoset, macaque, human, mouse, rat (g)
neurons = np.array([636e6, 6.38e9, 86e9, 71e6, 200e6])
slope = {}
slope["primate"] = ...
slope["rodent"] = ...
b, a = slope["rodent"], np.log10(neurons[3]) - slope["rodent"] * np.log10(mass[3])
print({k: round(float(v), 2) for k, v in slope.items()}, f";  a rodent with a 1.5 kg brain: about {10 ** (a + b * np.log10(1508)) / 1e9:.0f} billion neurons")

In [ ]:
import numpy as np
def test_slopes():
    "each slope is the log-log regression slope of neurons on mass within the group"
    assert abs(slope["primate"] - np.polyfit(np.log10(mass[:3]), np.log10(neurons[:3]), 1)[0]) < 1e-12
    assert abs(slope["rodent"] - np.log10(neurons[4] / neurons[3]) / np.log10(mass[4] / mass[3])) < 1e-12

def test_compare():
    "primates add neurons almost in proportion to mass (slope above 0.85); rodents more slowly (below 0.8)"
    assert slope["primate"] > 0.85 and slope["rodent"] < 0.8

test_slopes()
test_compare()
print("All checks passed")